# 13 - V2.4.1 Tail Selection Fix

Objectif:
- corriger la sous-modelisation de la queue
- sans repeter la regression RMSE globale de V2.4
- en appliquant une selection stricte et explicable.

Sorties dans `artifacts/v2_4_1_tail_recovery/`.


## 1) Cadrage: pourquoi V2.4 a choisi des candidats RMSE-regressifs

Constat:
- certains candidats ameliorent la queue (`q99_ratio_pos`, `rmse_top1pct`)
- mais degradent `rmse_prime`.

Decision:
- imposer des garde-fous hard avant toute selection robuste.


In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme(style="whitegrid")

NOTEBOOK_DIR = Path.cwd().resolve()
ROOT = NOTEBOOK_DIR
for _candidate in [NOTEBOOK_DIR, *NOTEBOOK_DIR.parents]:
    if (_candidate / "src").exists() and (_candidate / "data").exists():
        ROOT = _candidate
        break
else:
    raise RuntimeError("Project root not found. Expected folders: src/ and data/.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
ARTIFACTS_DIR = ROOT / "artifacts"

from src.insurance_pricing.experiments.quick import tail_selection as fx

RUN_PHASE_B_IF_NEEDED = True
MATERIALIZE_SUBMISSIONS = True

print("ROOT:", ROOT)


ROOT: c:\Users\icemo\Downloads\Calcul-prime-d-assurance


## 2) Chargement des artefacts V2.4


In [2]:
ctx = fx.load_v24_outputs(ROOT)
base = ctx["base_v24"]
cand_v24 = ctx["tail_candidates_multi_v24"].copy()
transform_store = ctx.get("tail_transform_store_v24", {})

print("Base run id:", base.get("base_run_id"))
print("Candidates V2.4 (multi):", cand_v24.shape)
display(cand_v24.head(10))


Base run id: base_v2|catboost|two_part_tweedie|cb_v2_c1|42|weighted_tail|isotonic|isotonic
Candidates V2.4 (multi): (25, 28)


,candidate_id,split,rmse_prime,rmse_prime_top1pct,q95_ratio_pos,q99_ratio_pos,body_rmse_proxy,tail_rmse_proxy,n_oof,n_oof_nonzero,...,tail_undercoverage_flag,selection_score_tail,pred_q90_oof,pred_q99_oof,pred_q90_test,pred_q99_test,q99_test_over_oof,q90_test_over_oof,std_test_over_oof,distribution_alignment_score
3,baseline_identity,multi,545.929482,4548.227515,0.476975,0.328630,100.615752,4548.227515,NaN,NaN,...,0.0,549.394258,158.623141,196.547049,164.156603,178.024449,0.905760,1.034884,0.736300,-1.644221
7,a1_rank_q09_l01_g10,multi,545.981209,4547.353253,0.501126,0.361493,101.887113,4547.353253,NaN,NaN,...,0.0,549.504445,159.057794,213.114458,164.423383,181.362789,0.851011,1.033734,0.731465,-2.195626
11,a1_rank_q09_l01_g20,multi,545.972649,4547.580703,0.489204,0.361493,101.587890,4547.580703,NaN,NaN,...,0.0,549.439695,158.623141,208.044183,164.423383,178.650459,0.858714,1.036566,0.728127,-2.139436
15,a1_rank_q09_l01_g30,multi,545.968011,4547.686701,0.483167,0.361493,101.444275,4547.686701,NaN,NaN,...,0.0,549.347987,158.623141,205.476889,164.264848,178.141839,0.866968,1.035567,0.727895,-2.052365
19,a1_rank_q09_l02_g10,multi,546.050660,4546.481539,0.525277,0.394356,103.234553,4546.481539,NaN,NaN,...,0.0,549.299512,160.124787,223.385286,164.423383,184.701129,0.826828,1.026845,0.725882,-2.414186
23,a1_rank_q09_l02_g20,multi,546.029319,4546.935758,0.501433,0.394356,102.621207,4546.935758,NaN,NaN,...,0.0,549.335303,158.623141,216.693467,164.423383,179.276470,0.827328,1.036566,0.719119,-2.471319
27,a1_rank_q09_l02_g30,multi,546.018248,4547.147491,0.489359,0.394356,102.327309,4547.147491,NaN,NaN,...,0.0,549.275608,158.623141,214.222727,164.373094,178.259230,0.832121,1.036249,0.718671,-2.422694
31,a1_rank_q09_l035_g10,multi,546.188055,4545.178749,0.561504,0.443650,105.391932,4545.178749,NaN,NaN,...,0.0,548.974633,162.901877,245.820128,164.694230,194.956086,0.793084,1.011003,0.716420,-2.691329
35,a1_rank_q09_l035_g20,multi,546.139632,4545.971845,0.519776,0.443650,104.281887,4545.971845,NaN,NaN,...,0.0,549.504998,159.165673,241.000502,164.423383,180.421915,0.748637,1.033033,0.704338,-3.270117
39,a1_rank_q09_l035_g30,multi,546.115548,4546.341689,0.498647,0.443650,103.750998,4546.341689,NaN,NaN,...,0.0,549.582582,158.623141,241.000502,164.423383,178.435315,0.740394,1.036566,0.703553,-3.371785


## 3) Audit identite / doublons


In [3]:
marked = fx.mark_identity_and_duplicate_candidates(cand_v24, transform_store, tol=1e-10)
cols = [c for c in ["candidate_id","candidate_family","is_identity_candidate","identity_reason","is_duplicate_candidate","duplicate_of","rmse_prime","q99_ratio_pos"] if c in marked.columns]
display(marked[cols].sort_values(["is_identity_candidate","is_duplicate_candidate","rmse_prime"], ascending=[False, False, True], na_position="last"))


,candidate_id,candidate_family,is_identity_candidate,identity_reason,is_duplicate_candidate,duplicate_of,rmse_prime,q99_ratio_pos
0,baseline_identity,baseline_identity,1,baseline_identity,0,,545.929482,0.328630
23,a2_mapper_q095_pwm_n150,tail_mapper_thresholded,1,metrics_equal_to_baseline,0,,545.929482,0.328630
24,a2_mapper_q095_pwm_n250,tail_mapper_thresholded,1,metrics_equal_to_baseline,0,,545.929482,0.328630
20,a2_mapper_q09_pwm_n250,tail_mapper_thresholded,0,,1,a2_mapper_q09_pwm_n150,546.502132,0.572874
21,a2_mapper_q09_iso_n150,tail_mapper_thresholded,0,,1,a2_mapper_q09_pwm_n150,546.502132,0.572874
22,a2_mapper_q09_iso_n250,tail_mapper_thresholded,0,,1,a2_mapper_q09_pwm_n150,546.502132,0.572874
15,a1_rank_q095_l01_g30,tail_scale_rank,0,,0,,545.959731,0.361493
14,a1_rank_q095_l01_g20,tail_scale_rank,0,,0,,545.963263,0.361493
3,a1_rank_q09_l01_g30,tail_scale_rank,0,,0,,545.968011,0.361493
13,a1_rank_q095_l01_g10,tail_scale_rank,0,,0,,545.969804,0.361493


## 4) Baseline anchoring et deltas


In [4]:
guarded = fx.compute_tail_guardrails(
    marked,
    baseline_id="baseline_identity",
    rmse_tol=0.15,
    q99_low=0.45,
    q99_high=0.70,
)
cols = [c for c in ["candidate_id","rmse_prime","rmse_delta_vs_baseline","q99_ratio_pos","rmse_prime_top1pct","top1pct_delta_vs_baseline"] if c in guarded.columns]
display(guarded[cols].sort_values("rmse_prime", na_position="last"))


,candidate_id,rmse_prime,rmse_delta_vs_baseline,q99_ratio_pos,rmse_prime_top1pct,top1pct_delta_vs_baseline
0,baseline_identity,545.929482,0.000000,0.328630,4548.227515,0.000000
23,a2_mapper_q095_pwm_n150,545.929482,0.000000,0.328630,4548.227515,0.000000
24,a2_mapper_q095_pwm_n250,545.929482,0.000000,0.328630,4548.227515,0.000000
15,a1_rank_q095_l01_g30,545.959731,0.030249,0.361493,4547.835191,-0.392324
14,a1_rank_q095_l01_g20,545.963263,0.033781,0.361493,4547.786303,-0.441212
3,a1_rank_q09_l01_g30,545.968011,0.038529,0.361493,4547.686701,-0.540815
13,a1_rank_q095_l01_g10,545.969804,0.040322,0.361493,4547.701055,-0.526460
2,a1_rank_q09_l01_g20,545.972649,0.043166,0.361493,4547.580703,-0.646812
1,a1_rank_q09_l01_g10,545.981209,0.051727,0.361493,4547.353253,-0.874262
18,a1_rank_q095_l02_g30,545.999463,0.069981,0.394356,4547.444174,-0.783341


## 5) Application des garde-fous hard


In [5]:
show_cols = [
    "candidate_id",
    "rmse_prime",
    "q99_ratio_pos",
    "meets_rmse_tol",
    "meets_gap_secondary",
    "meets_gap_aux",
    "meets_q99_low",
    "meets_q99_high",
    "meets_dist_penalty",
    "meets_not_identity",
    "meets_not_duplicate",
    "hard_admissible",
    "guardrail_fail_reasons",
]
show_cols = [c for c in show_cols if c in guarded.columns]
display(guarded[show_cols].sort_values(["hard_admissible","rmse_prime"], ascending=[False, True], na_position="last"))
print("n_admissible:", int((guarded.get("hard_admissible", 0).fillna(0).astype(int) == 1).sum()))


,candidate_id,rmse_prime,q99_ratio_pos,meets_rmse_tol,meets_gap_secondary,meets_gap_aux,meets_q99_low,meets_q99_high,meets_dist_penalty,meets_not_identity,meets_not_duplicate,hard_admissible,guardrail_fail_reasons
0,baseline_identity,545.929482,0.328630,1,1,1,0,1,1,0,1,0,"q99_too_low,identity"
23,a2_mapper_q095_pwm_n150,545.929482,0.328630,1,1,1,0,1,1,0,1,0,"q99_too_low,identity"
24,a2_mapper_q095_pwm_n250,545.929482,0.328630,1,1,1,0,1,1,0,1,0,"q99_too_low,identity"
15,a1_rank_q095_l01_g30,545.959731,0.361493,1,1,1,0,1,1,1,1,0,q99_too_low
14,a1_rank_q095_l01_g20,545.963263,0.361493,1,1,1,0,1,1,1,1,0,q99_too_low
3,a1_rank_q09_l01_g30,545.968011,0.361493,1,1,1,0,1,1,1,1,0,q99_too_low
13,a1_rank_q095_l01_g10,545.969804,0.361493,1,1,1,0,1,1,1,1,0,q99_too_low
2,a1_rank_q09_l01_g20,545.972649,0.361493,1,1,1,0,1,1,1,1,0,q99_too_low
1,a1_rank_q09_l01_g10,545.981209,0.361493,1,1,1,0,1,1,1,1,0,q99_too_low
18,a1_rank_q095_l02_g30,545.999463,0.394356,1,1,1,0,1,1,1,1,0,q99_too_low


n_admissible: 0


## 6) Tri des candidats selon le score V2.4.1


In [6]:
scored = fx.compute_selection_score_v241(guarded)
scored["phase_source"] = "phase_a"
display(
    scored[
        [c for c in ["candidate_id","candidate_family","rmse_prime","q99_ratio_pos","rmse_prime_top1pct","selection_score_v241","hard_admissible"] if c in scored.columns]
    ].sort_values(["hard_admissible","selection_score_v241","rmse_prime"], ascending=[False, True, True], na_position="last")
)


,candidate_id,candidate_family,rmse_prime,q99_ratio_pos,rmse_prime_top1pct,selection_score_v241,hard_admissible
15,a1_rank_q095_l01_g30,tail_scale_rank,545.959731,0.361493,4547.835191,547.409400,0
14,a1_rank_q095_l01_g20,tail_scale_rank,545.963263,0.361493,4547.786303,547.413197,0
0,baseline_identity,baseline_identity,545.929482,0.328630,4548.227515,547.416403,0
23,a2_mapper_q095_pwm_n150,tail_mapper_thresholded,545.929482,0.328630,4548.227515,547.416403,0
24,a2_mapper_q095_pwm_n250,tail_mapper_thresholded,545.929482,0.328630,4548.227515,547.416403,0
13,a1_rank_q095_l01_g10,tail_scale_rank,545.969804,0.361493,4547.701055,547.424239,0
3,a1_rank_q09_l01_g30,tail_scale_rank,545.968011,0.361493,4547.686701,547.439632,0
18,a1_rank_q095_l02_g30,tail_scale_rank,545.999463,0.394356,4547.444174,547.450119,0
17,a1_rank_q095_l02_g20,tail_scale_rank,546.007374,0.394356,4547.346511,547.458504,0
2,a1_rank_q09_l01_g20,tail_scale_rank,545.972649,0.361493,4547.580703,547.461075,0


## 7) Trigger conditionnel Phase B


In [7]:
ctx["guarded_candidates_v241"] = scored
phase_b_df = pd.DataFrame()
if RUN_PHASE_B_IF_NEEDED and int((scored.get("hard_admissible", 0).fillna(0).astype(int) == 1).sum()) == 0:
    phase_b_df = fx.run_phase_b_if_needed(ctx, base.get("base_run_row", {}), trigger_no_candidate=True)
    if not phase_b_df.empty:
        phase_b_df = fx.compute_tail_guardrails(
            phase_b_df,
            baseline_id="baseline_identity",
            rmse_tol=0.15,
            q99_low=0.45,
            q99_high=0.70,
        )
        phase_b_df = fx.compute_selection_score_v241(phase_b_df)
        phase_b_df["phase_source"] = "phase_b"
else:
    print("Phase B skipped: at least one admissible candidate in Phase A.")

print("Phase B rows:", len(phase_b_df))
if len(phase_b_df):
    display(phase_b_df)


Phase B rows: 2


,candidate_id,candidate_family,split,base_run_id,engine,family,config_id,feature_set,severity_mode,calibration,...,meets_q99_low,meets_q99_high,meets_dist_penalty,meets_tail_over,meets_not_identity,meets_not_duplicate,hard_admissible,guardrail_fail_reasons,selection_score_v241,phase_source
0,phaseb_wtail_b05_w10_q09_base_v2,sev_retrain_weighted_tail,multi,base_v2|catboost|two_part_tweedie|cb_v2_c1|42|...,catboost,two_part_tweedie,cb_v2_c1,base_v2,weighted_tail,isotonic,...,1,0,0,0,1,1,0,"q99_too_high,dist_penalty,tail_over",563.349949,phase_b
1,phaseb_wtail_b10_w10_q09_robust_v2,sev_retrain_weighted_tail,multi,base_v2|catboost|two_part_tweedie|cb_v2_c1|42|...,catboost,two_part_tweedie,cb_v2_c1,robust_v2,weighted_tail,isotonic,...,1,0,0,0,1,1,0,"q99_too_high,dist_penalty,tail_over",563.349949,phase_b


## 8) Selection finale robust vs lb_challenger


In [8]:
all_candidates = pd.concat([scored, phase_b_df], ignore_index=True, sort=False) if len(phase_b_df) else scored.copy()
all_candidates = all_candidates.drop_duplicates(subset=["candidate_id"], keep="last")

selected = fx.select_robust_and_challenger_v241(all_candidates)
role_map = selected.set_index("candidate_id")["role"].to_dict() if len(selected) else {}
all_candidates["selection_status"] = all_candidates["candidate_id"].map(role_map).fillna("rejected")

pareto = fx.tr.build_tail_pareto_front(all_candidates)
display(selected)
display(pareto)


,candidate_id,split,rmse_prime,rmse_prime_top1pct,q95_ratio_pos,q99_ratio_pos,body_rmse_proxy,tail_rmse_proxy,n_oof,n_oof_nonzero,...,seed,tweedie_power,phase_b_beta,phase_b_w_max,phase_b_threshold_q,phase_b_generated_from_benchmark,selection_reason,role,risk_tag,tail_strength_rank
0,baseline_identity,multi,545.929482,4548.227515,0.476975,0.328630,100.615752,4548.227515,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,fallback_baseline_no_admissible_tail_candidate,robust,robust,NaN
1,a2_mapper_q09_pwm_n150,multi,546.502132,4543.569310,0.602693,0.572874,108.739164,4543.569310,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,best_tail_candidate_under_soft_rmse_constraint,lb_challenger,public_private_risk,-0.572874


,candidate_id,split,rmse_prime,rmse_prime_top1pct,q95_ratio_pos,q99_ratio_pos,body_rmse_proxy,tail_rmse_proxy,n_oof,n_oof_nonzero,...,severity_mode,calibration,tail_mapper,seed,tweedie_power,phase_b_beta,phase_b_w_max,phase_b_threshold_q,phase_b_generated_from_benchmark,pareto_tail_distance
0,baseline_identity,multi,545.929482,4548.227515,0.476975,0.328630,100.615752,4548.227515,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.271370
1,a2_mapper_q095_pwm_n150,multi,545.929482,4548.227515,0.476975,0.328630,100.615752,4548.227515,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.271370
2,a2_mapper_q095_pwm_n250,multi,545.929482,4548.227515,0.476975,0.328630,100.615752,4548.227515,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.271370
3,a1_rank_q095_l01_g30,multi,545.959731,4547.835191,0.476975,0.361493,101.232670,4547.835191,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.238507
4,a1_rank_q095_l01_g20,multi,545.963263,4547.786303,0.476983,0.361493,101.307238,4547.786303,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.238507
5,a1_rank_q09_l01_g30,multi,545.968011,4547.686701,0.483167,0.361493,101.444275,4547.686701,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.238507
6,a1_rank_q09_l01_g20,multi,545.972649,4547.580703,0.489204,0.361493,101.587890,4547.580703,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.238507
7,a1_rank_q09_l01_g10,multi,545.981209,4547.353253,0.501126,0.361493,101.887113,4547.353253,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.238507
8,a1_rank_q095_l02_g30,multi,545.999463,4547.444174,0.476975,0.394356,101.895635,4547.444174,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.205644
9,a1_rank_q095_l02_g20,multi,546.007374,4547.346511,0.476990,0.394356,102.048198,4547.346511,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.205644


## 9) Export soumissions + artefacts V2.4.1


In [9]:
out_dir = fx.ensure_v241_dir(ROOT)
fx.tr._drop_array_cols_for_csv(all_candidates).to_csv(out_dir / "tail_candidates_registry_v2_4_1.csv", index=False)
fx.tr._drop_array_cols_for_csv(pareto).to_csv(out_dir / "tail_pareto_front_v2_4_1.csv", index=False)
fx.tr._drop_array_cols_for_csv(all_candidates).to_csv(out_dir / "tail_selection_report_v2_4_1.csv", index=False)

outputs = {"submission_paths": {}, "pred_audits": pd.DataFrame(), "generated_rows": pd.DataFrame()}
if MATERIALIZE_SUBMISSIONS:
    outputs = fx.materialize_submissions_v241(
        ctx,
        selected_df=selected,
        transform_store=transform_store,
        out_dir=out_dir,
        base_run_row=base.get("base_run_row", {}),
    )

display(outputs.get("pred_audits", pd.DataFrame()))
print("submission_paths:", outputs.get("submission_paths", {}))


,role,candidate_id,run_id,split,sample,n,pred_mean,pred_std,pred_q50,pred_q90,...,pred_share_nonzero,pred_identical_share,pred_identical_share_nonzero,pred_q99_q90_ratio,pred_n_nonzero,pred_q90_nonzero,pred_q99_nonzero,pred_q99_q90_ratio_nonzero,collapse_use_nonzero_support,distribution_collapse_flag
0,robust,baseline_identity,baseline_identity,test,test,50000,83.892877,124.212128,52.069757,174.953409,...,1.0,0.02328,0.02328,3.617727,50000,174.953409,632.933597,3.617727,0,0
1,lb_challenger,a2_mapper_q09_pwm_n150,a2_mapper_q09_pwm_n150,test,test,50000,226.818510,657.416537,52.147584,541.565162,...,1.0,0.02328,0.02328,6.158605,50000,541.565162,3335.286092,6.158605,0,0


submission_paths: {'robust': WindowsPath('c:/Users/icemo/Downloads/Calcul-prime-d-assurance/artifacts/v2_4_1_tail_recovery/submission_v2_4_1_robust.csv'), 'lb_challenger': WindowsPath('c:/Users/icemo/Downloads/Calcul-prime-d-assurance/artifacts/v2_4_1_tail_recovery/submission_v2_4_1_lb_challenger.csv')}


## 10) Rapport final: quoi soumettre et pourquoi


In [10]:
decision_path = fx.write_decision_report_v241(
    base_info=base,
    candidates_df=all_candidates,
    pareto_df=pareto,
    selected_df=selected,
    out_dir=fx.ensure_v241_dir(ROOT),
)
print("saved:", decision_path)
print(decision_path.read_text(encoding="utf-8")[:4000])


saved: c:\Users\icemo\Downloads\Calcul-prime-d-assurance\artifacts\v2_4_1_tail_recovery\submission_decision_v2_4_1.md
# Submission decision V2.4.1 Tail Selection Fix

## 1) Context
- Base run V2 anchor: `base_v2|catboost|two_part_tweedie|cb_v2_c1|42|weighted_tail|isotonic|isotonic`
- Goal: recover tail without uncontrolled global RMSE regression.

## 2) Hard guardrails
- rmse_prime <= baseline + 0.15
- q99_ratio_pos in [0.45, 0.70]
- secondary/aux gaps <= 1.0
- distribution_alignment_penalty <= 3.0
- no identity / duplicate candidates

## 3) Candidate summary

| candidate_id                       | candidate_family          |   rmse_prime |   rmse_delta_vs_baseline |   q99_ratio_pos |   rmse_prime_top1pct |   hard_admissible | guardrail_fail_reasons              |   selection_score_v241 |
|:-----------------------------------|:--------------------------|-------------:|-------------------------:|----------------:|---------------------:|------------------:|:------------------------------

## Checks obligatoires

- baseline_identity present
- robust respecte RMSE tol ou fallback baseline
- q99 robuste dans [0.45, 0.70] ou fallback baseline explicite
- submissions au format `index,pred` (50000 lignes, non negatif)


In [11]:
checks = []

checks.append({"check":"baseline_identity_exists", "ok": bool((all_candidates["candidate_id"].astype(str) == "baseline_identity").any())})

if len(selected):
    robust = selected[selected["role"].astype(str) == "robust"].iloc[0]
    baseline = all_candidates[all_candidates["candidate_id"].astype(str) == "baseline_identity"].iloc[0]
    rmse_ok = float(robust.get("rmse_prime", np.inf)) <= float(baseline.get("rmse_prime", np.inf)) + 0.15 + 1e-12
    is_fallback = str(robust.get("candidate_id")) == "baseline_identity"
    checks.append({"check":"robust_rmse_rule_or_fallback", "ok": bool(rmse_ok or is_fallback)})

    q99 = pd.to_numeric(pd.Series([robust.get("q99_ratio_pos")]), errors="coerce").iloc[0]
    q99_ok = (q99 >= 0.45 and q99 <= 0.70) if np.isfinite(q99) else False
    checks.append({"check":"robust_q99_rule_or_fallback", "ok": bool(q99_ok or is_fallback)})

for role, path in (outputs.get("submission_paths", {}) or {}).items():
    p = Path(path)
    ok = p.exists()
    checks.append({"check": f"{role}_exists", "ok": bool(ok)})
    if ok:
        sub = pd.read_csv(p)
        checks.append({"check": f"{role}_shape_50000", "ok": bool(len(sub) == 50000)})
        checks.append({"check": f"{role}_columns", "ok": bool(list(sub.columns) == ["index", "pred"])})
        checks.append({"check": f"{role}_pred_nonneg", "ok": bool((pd.to_numeric(sub["pred"], errors="coerce").fillna(-1) >= 0).all())})

checks_df = pd.DataFrame(checks)
display(checks_df)
print("all_checks_passed:", bool(checks_df["ok"].all()) if len(checks_df) else True)


,check,ok
0,baseline_identity_exists,True
1,robust_rmse_rule_or_fallback,True
2,robust_q99_rule_or_fallback,True
3,robust_exists,True
4,robust_shape_50000,True
5,robust_columns,True
6,robust_pred_nonneg,True
7,lb_challenger_exists,True
8,lb_challenger_shape_50000,True
9,lb_challenger_columns,True


all_checks_passed: True
